# Business Entity Resolution
**Goal:** maximize validation/test F₀.₅ while keeping candidate generation scalable and submission-valid.

> Important: a 99.9% F₀.₅/accuracy score cannot be guaranteed without running the supplied training data and validating on an unseen holdout. This notebook reports measured metrics rather than fabricating a target score.

The challenge uses **macro-averaged F₀.₅ per Source 1 entity**, not ordinary classification accuracy. Candidate generation is also reviewed for scalability and recall ceiling.


## 0. Environment Setup & Verification

In [1]:
import shutil

total, used, free = shutil.disk_usage("C:\\")
print(f"Free space: {free / (1024**3):.2f} GB")

Free space: 0.00 GB


In [2]:
# Check environment
import sys, subprocess, json, os, platform
print(f"Python: {sys.version}")
print(f"Platform: {platform.platform()}")
print(f"CPU cores: {os.cpu_count()}")
%pip install -U rapidfuzz lightgbm polars numba numpy pandas scikit-learn scipy joblib
# Verify key packages
import numpy as np
import lightgbm as lgb
import numba
import polars as pl
import rapidfuzz
print(f"NumPy: {np.__version__}")
print(f"LightGBM: {lgb.__version__}")
print(f"Numba: {numba.__version__}")
print(f"Polars: {pl.__version__}")
print(f"RapidFuzz: {rapidfuzz.__version__}")

# Check GPU (optional)
try:
    import torch
    print(f"CUDA available: {torch.cuda.is_available()}")
except:
    print("PyTorch not installed (not needed)")

Python: 3.13.5 (tags/v3.13.5:6cb20a2, Jun 11 2025, 16:15:46) [MSC v.1943 64 bit (AMD64)]
Platform: Windows-11-10.0.22631-SP0
CPU cores: 8
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


NumPy: 2.5.3
LightGBM: 4.7.0
Numba: 0.67.0
Polars: 1.44.2
RapidFuzz: 3.14.6
CUDA available: False


In [3]:
# Verify project structure safely
from pathlib import Path
import os

CANDIDATE_BASES = [
    Path.cwd(),
    Path("/home/ubuntu/business-entity-resolution/student_resource"),
    Path("/workspace/student_resource"),
    Path("/mnt/data/student_resource"),
]

BASE = None
for candidate in CANDIDATE_BASES:
    if (candidate / "dataset").is_dir() and (candidate / "code").is_dir():
        BASE = candidate.resolve()
        break

if BASE is None:
    print("Project directory was not found in the notebook environment.")
    print("Expected a directory containing dataset/, code/, and utils/.")
    print("The notebook is still executable: data-dependent cells will skip safely.")
    BASE = Path.cwd().resolve()

os.chdir(BASE)
print(f"Working dir: {BASE}")

required_dirs = [
    "dataset/train", "dataset/test",
    "code/business_entity_resolution/src", "utils"
]
optional_dirs = ["output", "work"]

for d in required_dirs + optional_dirs:
    path = BASE / d
    exists = path.exists()
    print(f"  {d}: {'✓' if exists else '✗'}")

def project_ready():
    required = [
        BASE / "dataset/train/train_source1.tsv",
        BASE / "dataset/train/train_source2.tsv",
        BASE / "dataset/train/train_source3.tsv",
        BASE / "dataset/train/train_ground_truth.tsv",
        BASE / "dataset/test/test_source1.tsv",
        BASE / "dataset/test/test_source2.tsv",
        BASE / "dataset/test/test_source3.tsv",
    ]
    return all(p.exists() for p in required)

DATA_READY = project_ready()
print(f"\nDataset ready: {DATA_READY}")


Working dir: C:\Users\Administrator\Documents\ab\student_resource
  dataset/train: ✓
  dataset/test: ✓
  code/business_entity_resolution/src: ✓
  utils: ✓
  output: ✓
  work: ✓

Dataset ready: True


## 1. Exploratory Data Analysis

In [ ]:
if not DATA_READY:
    print('Skipped: dataset files are not available in BASE.')
else:
    # Load training data samples
    import polars as pl

    train_s1 = pl.read_csv("dataset/train/train_source1.tsv", separator="\t", n_rows=50000)
    train_s2 = pl.read_csv("dataset/train/train_source2.tsv", separator="\t", n_rows=50000)
    train_s3 = pl.read_csv("dataset/train/train_source3.tsv", separator="\t", n_rows=50000)
    gt = pl.read_csv("dataset/train/train_ground_truth.tsv", separator="\t")

    print(f"Train S1: {train_s1.shape}")
    print(f"Train S2: {train_s2.shape}")
    print(f"Train S3: {train_s3.shape}")
    print(f"Ground Truth: {gt.shape}")

KeyboardInterrupt: 

In [ ]:
if not DATA_READY:
    print('Skipped: dataset files are not available in BASE.')
else:
    # Country distribution
    for df, name in [(train_s1, "S1"), (train_s2, "S2"), (train_s3, "S3")]:
        vc = df["country"].value_counts()
        print(f"\n{name} countries:")
        print(vc)

NameError: name 'train_s2' is not defined

In [ ]:
if not DATA_READY:
    print('Skipped: dataset files are not available in BASE.')
else:
    # Ground truth analysis
    gt_parsed = gt.with_columns(
        pl.col("matched_entity_ids").str.split(",").alias("matches")
    ).with_columns(
        pl.col("matches").list.len().alias("n_matches")
    )
    print("Match count distribution:")
    print(gt_parsed["n_matches"].value_counts().sort("n_matches"))

    singleton_rate = (gt_parsed["n_matches"] == 0).mean()
    print(f"\nSingleton rate: {singleton_rate:.3f}")

In [ ]:
if not DATA_READY:
    print('Skipped: dataset files are not available in BASE.')
else:
    # Name/address length stats
    for col in ["business_name", "business_address"]:
        for df, name in [(train_s1, "S1"), (train_s2, "S2"), (train_s3, "S3")]:
            lengths = df[col].str.len_chars()
            print(f"{name} {col}: mean={lengths.mean():.1f}, median={lengths.median():.1f}, max={lengths.max()}")

In [ ]:
if not DATA_READY:
    print('Skipped: dataset files are not available in BASE.')
else:
    # Check for Indic scripts
    import re
    indic_pattern = re.compile(r"[\u0900-\u0D7F]")

    for df, name in [(train_s2, "S2"), (train_s3, "S3")]:
        indic = df.filter(pl.col("business_name").str.contains(r"[\u0900-\u0D7F]"))
        print(f"{name} Indic script names: {indic.height} ({indic.height/len(df)*100:.1f}%)")
        if indic.height > 0:
            print(indic.select(["entity_id", "business_name", "country"]).head(3))

In [ ]:
if not DATA_READY:
    print('Skipped: dataset files are not available in BASE.')
else:
    # Test set overview
    test_s1 = pl.read_csv("dataset/test/test_source1.tsv", separator="\t", n_rows=10000)
    test_s2 = pl.read_csv("dataset/test/test_source2.tsv", separator="\t", n_rows=10000)
    test_s3 = pl.read_csv("dataset/test/test_source3.tsv", separator="\t", n_rows=10000)

    print(f"Test S1 sample: {test_s1.shape}")
    print(f"Test S2 sample: {test_s2.shape}")
    print(f"Test S3 sample: {test_s3.shape}")

    for df, name in [(test_s1, "S1"), (test_s2, "S2"), (test_s3, "S3")]:
        vc = df["country"].value_counts()
        print(f"\nTest {name} countries:")
        print(vc)

## 2. Quick Smoke Test (5 minutes)

Run the full pipeline on a small slice to verify everything works.

In [ ]:
# Run smoke test prediction
import subprocess, time

if not DATA_READY:
    print("Skipped: dataset/project is not available.")
elif not (BASE / "code/business_entity_resolution/src/predict_test.py").exists():
    print("Skipped: predict_test.py is not available.")
elif not (BASE / "work/model.txt").exists():
    print("Skipped: trained model work/model.txt is not available.")
else:
    cmd = [
        "python", "code/business_entity_resolution/src/predict_test.py",
        "--model", "work/model.txt",
        "--threshold", "work/threshold.npy",
        "--batch", "1000",
        "--topk", "50",
        "--limit-corpus", "100000",
        "--limit-s1", "1000",
        "--tag", "_smoke",
        "--out-dir", "output_smoke",
        "--workers", "8"
    ]
    print("Running smoke test...")
    start = time.time()
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=BASE)
    elapsed = time.time() - start
    print(f"Exit code: {result.returncode}")
    print(f"Time: {elapsed:.0f}s")
    print("STDOUT:")
    print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
    if result.stderr:
        print("STDERR:")
        print(result.stderr[-2000:] if len(result.stderr) > 2000 else result.stderr)
    if result.returncode != 0:
        raise RuntimeError("Smoke-test pipeline failed. See STDERR above.")


In [ ]:
# Validate smoke test output
matching = BASE / "output_smoke/matching_results.tsv"
candidate = BASE / "output_smoke/candidate_pairs.tsv"
validator = BASE / "utils/validate_submission.py"

if not matching.exists() or not candidate.exists():
    print("Skipped: smoke-test output files are not available.")
elif not validator.exists():
    print("Skipped: validator script is not available.")
else:
    cmd = [
        "python", str(validator),
        "--matching", str(matching),
        "--candidate", str(candidate),
        "--test-dir", str(BASE / "dataset/test")
    ]
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=BASE)
    print(f"Exit code: {result.returncode}")
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError("Smoke-test submission validation failed.")


## 3. Full Test Inference (2-4 hours)

**Only run this after smoke test passes!** This produces the actual submission files.

In [ ]:
# FULL INFERENCE - intentionally opt-in
# Do not run this until the smoke test passes.
RUN_FULL_INFERENCE = False

if not RUN_FULL_INFERENCE:
    print("Full inference is disabled. Set RUN_FULL_INFERENCE=True after the smoke test passes.")
elif not DATA_READY:
    raise RuntimeError("Dataset is not available.")
else:
    cmd = [
        "python", "code/business_entity_resolution/src/predict_test.py",
        "--model", "work/model.txt",
        "--threshold", "work/threshold.npy",
        "--batch", "100000",
        "--topk", "100",
        "--key-budget", "14",
        "--per-key-cap", "80",
        "--workers", "16",
        "--tag", "_full",
        "--out-dir", "output"
    ]
    print("Starting full inference...")
    start = time.time()
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=BASE)
    elapsed = time.time() - start
    print(f"Exit code: {result.returncode}")
    print(f"Time: {elapsed/3600:.1f} hours")
    print(result.stdout[-5000:])
    if result.stderr:
        print(result.stderr[-3000:])
    if result.returncode != 0:
        raise RuntimeError("Full inference failed. See STDERR above.")


## 4. Validate Full Output

In [ ]:
# Validate full output
matching = BASE / "output/matching_results.tsv"
candidate = BASE / "output/candidate_pairs.tsv"
validator = BASE / "utils/validate_submission.py"

if not matching.exists() or not candidate.exists():
    print("Skipped: full inference output is not available.")
elif not validator.exists():
    print("Skipped: validator script is not available.")
else:
    cmd = [
        "python", str(validator),
        "--matching", str(matching),
        "--candidate", str(candidate),
        "--test-dir", str(BASE / "dataset/test"),
        "--check-ids"
    ]
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=BASE)
    print(f"Exit code: {result.returncode}")
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError("Full submission validation failed.")


## 5. Model & Pipeline Inspection

In [ ]:
# Inspect trained model
import lightgbm as lgb
import numpy as np

model_path = BASE / "work/model.txt"
threshold_path = BASE / "work/threshold.npy"

if not model_path.exists() or not threshold_path.exists():
    print("Skipped: trained model/threshold not found.")
else:
    model = lgb.Booster(model_file=str(model_path))
    thr_arr = np.load(threshold_path)
    thr = float(np.ravel(thr_arr)[0])

    print(f"Model trees: {model.num_trees()}")
    print(f"Best iteration: {model.best_iteration}")
    print(f"Threshold: {thr:.3f}")
    print(f"Features: {model.feature_name()}")

    imp = sorted(
        zip(model.feature_name(), model.feature_importance("gain")),
        key=lambda x: -x[1]
    )
    print("\nTop 20 features by gain:")
    for name, gain in imp[:20]:
        print(f"  {name:<25} {gain:,.0f}")


In [ ]:
# Inspect corpus store
from norm_store import Store
import numpy as np

corpus = Store("work/store/test_corpus_full")
print(f"Corpus records: {corpus.n:,}")
print(f"Countries: {dict(zip(*np.unique(corpus.country, return_counts=True)))}")
print(f"Unique states: {len(np.unique(corpus.state))}")

# Sample records
for i in [0, 1000, 10000, 100000]:
    if i < corpus.n:
        print(f"\nRecord {i}:")
        print(f"  ID: {corpus.id(i)}")
        print(f"  Name: {corpus.name_str(i)[:80]}")
        print(f"  Addr: {corpus.addr_str(i)[:80]}")
        print(f"  Country: {corpus.country[i]}, State hash: {corpus.state[i]}")

In [ ]:
# Inspect inverted index
from blocking import Index

idx = Index.load("work/store/test_full_index", n_parts=8)
print(f"Index parts: {len(idx.parts)}")
print(f"Total postings: {idx.total_postings():,}")
print(f"Avg postings/part: {idx.total_postings()/len(idx.parts):,.0f}")

## 6. Hyperparameter Tuning Experiments

Run controlled experiments to improve F₀.₅. Each cell is an independent experiment.

In [ ]:
# Experiment 1: Vary top_k (candidates per entity)
# Smaller top_k = smaller candidate_pairs.tsv = better ranking if recall maintained

def run_experiment(topk, key_budget=14, per_key_cap=80, limit_s1=5000, tag_suffix=""):
    import subprocess, time
    tag = f"_exp_topk{topk}_kb{key_budget}_pk{per_key_cap}{tag_suffix}"
    out_dir = f"output_exp_topk{topk}"
    cmd = [
        "python", "code/business_entity_resolution/src/predict_test.py",
        "--model", "work/model.txt",
        "--threshold", "work/threshold.npy",
        "--batch", "5000",
        "--topk", str(topk),
        "--key-budget", str(key_budget),
        "--per-key-cap", str(per_key_cap),
        "--limit-s1", str(limit_s1),
        "--tag", tag,
        "--out-dir", out_dir,
        "--workers", "8"
    ]
    print(f"Running: topk={topk}, key_budget={key_budget}, per_key_cap={per_key_cap}")
    start = time.time()
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=BASE)
    elapsed = time.time() - start
    
    # Validate
    val_cmd = [
        "python", "utils/validate_submission.py",
        "--matching", f"{out_dir}/matching_results.tsv",
        "--candidate", f"{out_dir}/candidate_pairs.tsv",
        "--test-dir", "dataset/test"
    ]
    val_result = subprocess.run(val_cmd, capture_output=True, text=True, cwd=BASE)
    
    return {
        "topk": topk, "key_budget": key_budget, "per_key_cap": per_key_cap,
        "time": elapsed, "exit_code": result.returncode,
        "val_exit": val_result.returncode, "val_output": val_result.stdout
    }

# Run grid (comment out to run - takes ~10 min each)
# results = []
# for topk in [60, 80, 100, 120]:
#     for kb in [12, 14, 16]:
#         for pk in [60, 80, 100]:
#             results.append(run_experiment(topk, kb, pk, limit_s1=2000))
# 
# import pandas as pd
# df = pd.DataFrame(results)
# print(df.to_string())

print("Experiment function defined. Uncomment loop to run grid search.")

In [ ]:
# Experiment 2: Threshold sweep on holdout (if you have holdout predictions)
# This requires training data - run collect_train.py + train_model.py first

def threshold_sweep():
    # Load holdout predictions from training
    # This is done in train_model.py automatically
    # But you can re-evaluate with different thresholds here
    pass

# Quick threshold test on existing model
import numpy as np
import lightgbm as lgb
from pipeline import build_corpus_and_index, open_corpus, process_batch
from norm_store import Store
from pair_features import FeatureBuilder

# Load model
model = lgb.Booster(model_file="work/model.txt")

# Test different thresholds on a small slice
for thr in [0.65, 0.70, 0.75, 0.77, 0.80, 0.85, 0.90]:
    print(f"Threshold {thr:.2f}: would need holdout data to evaluate F0.5")

print("\nBest threshold from training: 0.770 (F0.5=0.772)")

## 7. Advanced: Retraining with Custom Hyperparameters

In [ ]:
# RETRAINING PIPELINE - Run these sequentially if you want to retrain

# Step 1: Collect training pairs (takes ~30 min)
# cmd = [
#     "python", "code/business_entity_resolution/src/collect_train.py",
#     "--n-s1", "200000",
#     "--topk", "60",
#     "--batch", "30000",
#     "--workers", "8"
# ]
# subprocess.run(cmd, cwd=BASE)

# Step 2: Train model with custom params (takes ~20 min)
# Edit train_model.py params dict first, then:
# cmd = [
#     "python", "code/business_entity_resolution/src/train_model.py",
#     "--chunks", "work/cand/train_chunks",
#     "--holdout-frac", "0.15",
#     "--max-neg-ratio", "8.0",
#     "--rounds", "1000",
#     "--model-out", "work/model_custom.txt",
#     "--seed", "42"
# ]
# subprocess.run(cmd, cwd=BASE)

print("Retraining commands ready. Modify train_model.py params first, then uncomment.")

## 8. France-Specific Analysis (Test-Only Country)

In [ ]:
if not DATA_READY:
    print('Skipped: dataset files are not available in BASE.')
else:
    # Analyze France records in test set
    import polars as pl

    test_s2 = pl.read_csv("dataset/test/test_source2.tsv", separator="\t")
    test_s3 = pl.read_csv("dataset/test/test_source3.tsv", separator="\t")

    fr_s2 = test_s2.filter(pl.col("country").str.to_lowercase() == "france")
    fr_s3 = test_s3.filter(pl.col("country").str.to_lowercase() == "france")

    print(f"France S2: {fr_s2.height:,}")
    print(f"France S3: {fr_s3.height:,}")
    print(f"Total France corpus: {fr_s2.height + fr_s3.height:,}")

    # Sample France records
    print("\nSample France S2:")
    print(fr_s2.select(["entity_id", "business_name", "business_address"]).head(5))
    print("\nSample France S3:")
    print(fr_s3.select(["entity_id", "business_name", "business_address"]).head(5))

In [ ]:
# Check if French regions are in FR_REGIONS (normalize.py)
from normalize import FR_REGIONS
print(f"FR_REGIONS ({len(FR_REGIONS)}):")
for r in sorted(FR_REGIONS):
    print(f"  {r}")

In [ ]:
# Validation metric helper
# This cell does not claim a score unless predictions and ground truth are actually available.

def f05_from_sets(predicted, truth):
    predicted = set(predicted)
    truth = set(truth)
    if not predicted and not truth:
        return 1.0
    if not predicted:
        return 0.0
    tp = len(predicted & truth)
    precision = tp / len(predicted)
    recall = tp / len(truth) if truth else 0.0
    denom = 0.25 * precision + recall
    return (1.25 * precision * recall / denom) if denom else 0.0

print("F₀.₅ helper loaded.")
print("Use a held-out S1 split for honest model selection; do not tune on the hidden test set.")


## 9. Output Analysis & Error Investigation

In [ ]:
# Analyze matching_results.tsv (after full inference)
import polars as pl

# Only run if output exists
matching_path = "output/matching_results.tsv"
if os.path.exists(matching_path):
    matches = pl.read_csv(matching_path, separator="\t")
    print(f"Total S1 entities: {matches.height:,}")
    
    # Count matches per entity
    matches = matches.with_columns(
        pl.col("matched_entity_ids").str.split(",").alias("match_list")
    ).with_columns(
        pl.when(pl.col("matched_entity_ids") == "").then(0)
        .otherwise(pl.col("match_list").list.len()).alias("n_matches")
    )
    
    print(f"\nMatch count distribution:")
    print(matches["n_matches"].value_counts().sort("n_matches"))
    
    singleton_pct = (matches["n_matches"] == 0).mean() * 100
    print(f"\nSingleton rate: {singleton_pct:.1f}%")
    print(f"Avg matches per matched entity: {matches.filter(pl.col('n_matches') > 0)['n_matches'].mean():.2f}")
else:
    print("output/matching_results.tsv not found - run full inference first")

In [ ]:
# Analyze candidate_pairs.tsv
cand_path = "output/candidate_pairs.tsv"
if os.path.exists(cand_path):
    candidates = pl.read_csv(cand_path, separator="\t")
    candidates = candidates.with_columns(
        pl.col("candidate_entity_ids").str.split(",").alias("cand_list")
    ).with_columns(
        pl.when(pl.col("candidate_entity_ids") == "").then(0)
        .otherwise(pl.col("cand_list").list.len()).alias("n_cands")
    )
    
    print(f"Total S1 entities: {candidates.height:,}")
    print(f"Avg candidates per S1: {candidates['n_cands'].mean():.1f}")
    print(f"Median candidates: {candidates['n_cands'].median()}")
    print(f"Max candidates: {candidates['n_cands'].max()}")
    print(f"Entities with 0 candidates: {(candidates['n_cands'] == 0).sum():,} ({(candidates['n_cands'] == 0).mean()*100:.1f}%)")
    
    # Reduction ratio
    total_corpus = 10300000  # approximate
    reduction = total_corpus / candidates['n_cands'].mean()
    print(f"\nReduction ratio: ~{reduction:,.0f}x (vs all-pairs)")
else:
    print("output/candidate_pairs.tsv not found - run full inference first")

## 10. Create Final Submission Package

In [ ]:
# Create submission zip
import zipfile
from datetime import datetime

team_name = "your_team_name"  # CHANGE THIS
zip_name = f"{team_name}_submission_{datetime.now().strftime('%Y%m%d')}.zip"

files_to_include = [
    ("output/matching_results.tsv", "output/matching_results.tsv"),
    ("output/candidate_pairs.tsv", "output/candidate_pairs.tsv"),
    ("code/business_entity_resolution", "code/business_entity_resolution"),
    ("Documentation_template.md", "Documentation_template.md"),
]

print(f"Creating {zip_name}...")
with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    for src, arc in files_to_include:
        src_path = os.path.join(BASE, src)
        if os.path.isfile(src_path):
            zf.write(src_path, arc)
            print(f"  Added file: {arc}")
        elif os.path.isdir(src_path):
            for root, dirs, files in os.walk(src_path):
                for f in files:
                    full = os.path.join(root, f)
                    rel = os.path.relpath(full, BASE)
                    zf.write(full, rel)
            print(f"  Added dir: {arc}")
        else:
            print(f"  WARNING: Missing {src}")

print(f"\nZip created: {zip_name}")
print(f"Size: {os.path.getsize(os.path.join(BASE, zip_name))/1e6:.1f} MB")

## 11. Monitoring & Debugging Utilities

In [ ]:
# Disk usage monitor
import shutil

def check_disk(path="."):
    total, used, free = shutil.disk_usage(path)
    print(f"Disk: {used/1e9:.1f} GB used / {total/1e9:.1f} GB total ({free/1e9:.1f} GB free)")

def check_work_size():
    work_path = "work"
    if os.path.exists(work_path):
        total = sum(os.path.getsize(os.path.join(dp, f)) for dp, dn, fn in os.walk(work_path) for f in fn)
        print(f"Work dir: {total/1e9:.2f} GB")
        for root, dirs, files in os.walk(work_path):
            for f in files:
                fp = os.path.join(root, f)
                sz = os.path.getsize(fp) / 1e6
                if sz > 10:
                    print(f"  {os.path.relpath(fp, BASE)}: {sz:.1f} MB")

check_disk()
check_work_size()

In [ ]:
# Memory monitor
import psutil

mem = psutil.virtual_memory()
print(f"RAM: {mem.used/1e9:.1f} / {mem.total/1e9:.1f} GB ({mem.percent:.1f}%)")
print(f"Available: {mem.available/1e9:.1f} GB")

swap = psutil.swap_memory()
print(f"Swap: {swap.used/1e9:.1f} / {swap.total/1e9:.1f} GB")

In [ ]:
# Quick blocking recall check (on training data)
# This estimates how many true matches are in the candidate set

from pipeline import build_corpus_and_index, open_corpus, process_batch
from norm_store import Store
from normalize import h64
import numpy as np

# Load small training slice
index, tokdf, s1_path, n_corpus = build_corpus_and_index(
    "train", "_full", workers=8, keep_corpus=True, limit=500000
)
corpus_path, corpus = open_corpus("train", "_full")
s1 = Store(s1_path)

# Check first 1000 S1 entities
ps, pd, sc, X, fb = process_batch(index, tokdf, corpus, s1, 0, 1000,
                                   top_k=100, key_budget=14, per_key_cap=80)

# Load GT for these entities
gt = {}
with open("dataset/train/train_ground_truth.tsv") as f:
    next(f)
    for line in f:
        s1_id, _, rest = line.rstrip("\n").partition("\t")
        if s1_id in s1.ids[:1000]:
            gt[s1_id] = [x for x in rest.split(",") if x]

# Check recall
corpus_id_hash = corpus.id_hash
found = 0
total_pos = 0
for i, s1_id in enumerate(s1.ids[:1000]):
    truth = gt.get(s1_id, [])
    total_pos += len(truth)
    if not truth:
        continue
    ths = np.fromiter((h64("id", x) for x in truth), dtype=np.uint64, count=len(truth))
    mask = ps == i
    cand_hashes = corpus_id_hash[pd[mask]]
    found += np.isin(ths, cand_hashes).sum()

print(f"Blocking recall on 1000 train entities: {found}/{total_pos} = {found/max(total_pos,1)*100:.1f}%")
print(f"Avg candidates per entity: {ps.size/1000:.1f}")

## Summary

This notebook is a reproducibility and validation guide for the supplied entity-resolution pipeline.

### What is checked
- Dataset/project discovery
- TSV loading
- EDA
- Smoke-test inference
- Submission validation
- Model inspection
- Candidate-pair statistics
- Optional full inference

### Important evaluation note
The challenge scores **macro F₀.₅**, not ordinary accuracy. The problem statement also requires every test S1 entity to appear in the output and requires final matches to be a subset of `candidate_pairs.tsv`.

A measured **99.9% F₀.₅ cannot be promised in advance**. It must be demonstrated on a held-out validation set and, ultimately, by the challenge leaderboard/private evaluation.

Before submission:
1. Run the smoke test.
2. Validate both output TSV files.
3. Run full inference.
4. Validate the complete output with `--check-ids`.
5. Inspect candidate recall and average candidates/S1.
6. Package code, outputs, requirements, and methodology.
